# state-dict-load — ex2: load_state_dict(strict=True) raises on missing keys — catch and parse

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `state-dict-load`. Running the final beacon cell reports progress against the `Transfer: state_dict load` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Transfer: state_dict load` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`state-dict-load`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "state-dict-load"
DD_SUBTOPIC = "Transfer: state_dict load"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `strict=True` raises on missing keys — catch + report

Ex1 used `strict=False` and inspected `_IncompatibleKeys`. The deepening move is to call with `strict=True` (the default) on a checkpoint that's missing some keys, catch the `RuntimeError`, and extract the missing key names from its message.

```python
try:
    model.load_state_dict(checkpoint)  # strict=True default
except RuntimeError as e:
    msg = str(e)
    # Parse 'Missing key(s) in state_dict:' section.
```

**Why catch instead of preventing.** If you control the checkpoint, you'd never see this. But when loading a CHECKPOINT THAT EVOLVED with the model (added new layers, renamed parameters), a `strict=True` load is the canary that warns you the checkpoint is stale before you train on stale weights.

**Format of the error message.** The `RuntimeError` text contains two sections: `'Missing key(s) in state_dict: "key1", "key2", ...'` and (separately) `'Unexpected key(s) in state_dict: ...'`. The quoted keys are comma-separated. Parsing them is a regex job, but for a small drill you can split on the markers.

**Trade-off vs `strict=False`.** `strict=False` returns the lists directly (cleaner), but silently allows mismatches if you forget to inspect the return. `strict=True` forces the caller to handle mismatches explicitly — the right default for production code.

### Exercise 2 — load_state_dict(strict=True) raises on missing keys — catch and parse

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the `RuntimeError` raised by `load_state_dict(strict=True)` on a checkpoint with missing entries, extracting the missing key names from the exception message into a sorted list.
> Keywords: load_state_dict, strict-true, RuntimeError, missing-keys
> ```

**KCs targeted:** `strict-true-raises-runtime-error`, `parse-missing-keys-from-error-message`

Implement `ex2_load_strict_and_report(model, checkpoint)`.

Call `model.load_state_dict(checkpoint)` (no `strict=` kwarg — the default IS `strict=True`). The checkpoint is missing some keys, so a `RuntimeError` will be raised.

Catch it and parse the missing key names. The error message contains a line of the form:
```
Missing key(s) in state_dict: "fc.weight", "fc.bias".
```
Extract the quoted keys (anything between `"` characters on that line). Return:
```
{
  'raised': True | False,
  'missing_keys': sorted list[str],
  'error_message': str (str(exc); empty string if no raise),
}
```

Use a regex like `re.findall(r'"([^"]+)"', message_after_marker)` to pull the keys. To avoid catching unexpected-keys quotes (which appear separately), grab the substring after `'Missing key(s) in state_dict:'` and ending at the next newline or the next section marker like `'Unexpected key(s)'`.

If `load_state_dict` does NOT raise (checkpoint is complete), set `raised=False`, `missing_keys=[]`, `error_message=''`.

In [ ]:
import re

def ex2_load_strict_and_report(model, checkpoint):
    try:
        model.load_state_dict(checkpoint)
        return {'raised': False, 'missing_keys': [], 'error_message': ''}
    except RuntimeError as e:
        msg = str(e)
        # Isolate the 'Missing key(s)...' section to avoid catching
        # 'Unexpected key(s)...' quoted names.
        missing = []
        marker = 'Missing key(s) in state_dict:'
        if marker in msg:
            tail = msg.split(marker, 1)[1]
            # Stop at the next section marker if present.
            for stop in ('Unexpected key(s)', 'size mismatch', 'Error(s)'):
                if stop in tail:
                    tail = tail.split(stop, 1)[0]
            missing = re.findall(r'"([^"]+)"', tail)
        return {
            'raised': True,
            'missing_keys': sorted(missing),
            'error_message': msg,
        }


<details><summary>Solution</summary>

```python
import re

def ex2_load_strict_and_report(model, checkpoint):
    try:
        model.load_state_dict(checkpoint)
        return {'raised': False, 'missing_keys': [], 'error_message': ''}
    except RuntimeError as e:
        msg = str(e)
        # Isolate the 'Missing key(s)...' section to avoid catching
        # 'Unexpected key(s)...' quoted names.
        missing = []
        marker = 'Missing key(s) in state_dict:'
        if marker in msg:
            tail = msg.split(marker, 1)[1]
            # Stop at the next section marker if present.
            for stop in ('Unexpected key(s)', 'size mismatch', 'Error(s)'):
                if stop in tail:
                    tail = tail.split(stop, 1)[0]
            missing = re.findall(r'"([^"]+)"', tail)
        return {
            'raised': True,
            'missing_keys': sorted(missing),
            'error_message': msg,
        }
```

**Section isolation matters.** `RuntimeError`'s message can contain BOTH 'Missing key(s)' and 'Unexpected key(s)' sections, each with quoted names. Naively running `re.findall(r'"([^"]+)"', msg)` would mix them. Splitting on the marker keeps the parser honest.

**Why `strict=True` is the production default.** A silent `strict=False` load can train on a checkpoint that's stale, head-mismatched, or partially restored. The exception forces explicit handling — even if your handler is just `load_state_dict(ckpt, strict=False)` after logging the diagnostic.

**Quoted-key parsing is fragile.** PyTorch's error format is stable but undocumented. A more robust solution uses `strict=False` first to get the `_IncompatibleKeys` structured return, then chooses to raise. The drill exercises the parse path explicitly because it's what you do when wrapping a library you don't control.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()